# Structured Output
<img src="./assets/LC_StructuredOutput.png" width="500">


## Setup

Load and/or check for needed environmental variables

In [2]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****ywsA
DEEPSEEK_API_KEY=****c1e4
DEEPSEEK_BASE_URL=****.com
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Structured Output Example

In [6]:
from typing_extensions import TypedDict

from langchain.agents import create_agent


# 定义结构化输出的目标格式
# 这里要求 agent 最后返回 name、email、phone 三个字段
class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str


# 创建 agent
# response_format=ContactInfo 表示:
# 不只是让模型输出普通文本,而是要求它按 ContactInfo 这套结构返回结果
#并且实际上是将ContactInfo包装为工具自动加入到tool列表中
#最后强制模型使用这个工具返回结构化结果
#不过这里好像是因为deepseek不支持结构化输出?
#
agent = create_agent(
    model="deepseek-chat",
    response_format=ContactInfo,
)

# 这里是一段待提取信息的对话文本
# 目标是让模型从自然语言里整理出联系人姓名、邮箱和电话
recorded_conversation = """We talked with John Doe. He works over at Example. His number is, let's see,
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

# 调用 agent
# messages 里放用户输入的文本内容
result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)
#看一下原始输出
print(result)
# structured_response 是结构化提取后的最终结果
# 一般会得到类似:
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '5551234567'}
result["structured_response"]


#原始输出
# {
#     'messages': [
#         HumanMessage(
#             content="We talked with John Doe. He works over at Example. His number is, let's see,\nfive, five, five, one two three, four, five, six seven. Did you get that?\nAnd, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch.用中文回答",
#             additional_kwargs={},
#             response_metadata={},
#             id='5999cc7d-618f-4935-94ee-639e284940a3'
#         ),
#         AIMessage(
#             content='',
#             additional_kwargs={
#                 'refusal': None
#             },
#             response_metadata={
#                 'token_usage': {
#                     'completion_tokens': 75,
#                     'prompt_tokens': 388,
#                     'total_tokens': 463,
#                     'completion_tokens_details': None,
#                     'prompt_tokens_details': {
#                         'audio_tokens': None,
#                         'cached_tokens': 320
#                     },
#                     'prompt_cache_hit_tokens': 320,
#                     'prompt_cache_miss_tokens': 68
#                 },
#                 'model_provider': 'deepseek',
#                 'model_name': 'deepseek-chat',
#                 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache',
#                 'id': '396a32e1-f015-4bb4-af99-038dcfd829af',
#                 'finish_reason': 'tool_calls',
#                 'logprobs': None
#             },
#             id='lc_run--749b0870-f9f3-4acf-8bef-0b5e3940dd78-0',
#             tool_calls=[
#                 {
#                     'name': 'ContactInfo',
#                     'args': {
#                         'name': 'John Doe',
#                         'email': 'john@example.com',
#                         'phone': '5551234567'
#                     },
#                     'id': 'call_00_qKMUqZ28cNoB2Pl6Rlvftw3M',
#                     'type': 'tool_call'
#                 }
#             ],
#             usage_metadata={
#                 'input_tokens': 388,
#                 'output_tokens': 75,
#                 'total_tokens': 463,
#                 'input_token_details': {
#                     'cache_read': 320
#                 },
#                 'output_token_details': {}
#             }
#         ),
#         ToolMessage(
#             content="Returning structured response: {'name': 'John Doe', 'email': 'john@example.com', 'phone': '5551234567'}",
#             name='ContactInfo',
#             id='2a024e00-06a5-4c8e-994d-7cde496be3af',
#             tool_call_id='call_00_qKMUqZ28cNoB2Pl6Rlvftw3M'
#         )
#     ],
#     'structured_response': {
#         'name': 'John Doe',
#         'email': 'john@example.com',
#         'phone': '5551234567'
#     }
# }

{'messages': [HumanMessage(content="We talked with John Doe. He works over at Example. His number is, let's see,\nfive, five, five, one two three, four, five, six seven. Did you get that?\nAnd, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch.", additional_kwargs={}, response_metadata={}, id='94014d80-7eda-4de2-ae0a-365cf46dc099'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 385, 'total_tokens': 460, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 65}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '2995f8e0-9920-4b54-8f65-309d0cd24b76', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--b4f692ef-0152-4007-bc5e-4b04cf1b0fb7-0', tool_calls=[{'name'

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '5551234567'}

#### Multiple data types are supported 

* pydantic `BaseModel`
* `TypedDict`
* `dataclasses`
* json schema (dict)

In [3]:
from langchain.agents import create_agent
from pydantic import BaseModel


#这里主要演示各种数据类型都能够起到结构化输出的效果
#不过最多的还是BaseModel?(或许)
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent = create_agent(model="deepseek-chat", response_format=ContactInfo)

recorded_conversation = """ We talked with John Doe. He works over at Example. His number is, let's see, 
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='5551234567')